# Chapter 2 — GPT-2 Tokenization and Language-Model Data Loading

This notebook develops the data pipeline that turns raw text into the input-target pairs used to train a GPT-style language model. It begins with OpenAI's `tiktoken` library and the GPT-2 encoding, then checks how ordinary text, an end-of-text marker, and an unfamiliar-looking word are represented as integer token IDs. Encoding and decoding are explored together so that the numerical representation never becomes detached from the text it represents.

The second half moves from tokenization to supervised training examples. The source text in `text-verdict.txt` is converted into one long token sequence, sliced into overlapping context windows, and shifted by one position to create next-token targets. A custom PyTorch `Dataset` stores those aligned windows, while `DataLoader` controls batching, shuffling, overlap, and incomplete batches. The preserved outputs make these relationships concrete by showing token IDs, decoded text, adjacent context-target pairs, and tensor batches.

The notebook is part of a personal educational implementation of Chapter 2 from Sebastian Raschka's *Build a Large Language Model (From Scratch)*. Its focus is understanding the mechanics of preparing text for language-model training, not training a model yet.

## Learning objectives

By the end of the notebook, you should be able to:

- encode and decode text with the GPT-2 tokenizer;
- explain why byte pair encoding can represent unfamiliar text as subword pieces;
- construct contexts and one-token-shifted targets;
- distinguish context length from stride and batch size;
- implement a PyTorch `Dataset` for sliding token windows;
- inspect the shapes and alignment of batched language-model examples.

| Stage | Input | Main operation | Output |
|---|---|---|---|
| Tokenization | Raw text | GPT-2 BPE encoding | Token IDs |
| Round trip | Token IDs | Decoding | Reconstructed text |
| Windowing | Long token sequence | Slice by context length and stride | Input windows |
| Target creation | Input windows | Shift by one token | Next-token targets |
| Batching | Dataset examples | PyTorch `DataLoader` | Input and target tensors |


## 1. Load the GPT-2 tokenizer

The notebook first imports `tiktoken`, records the installed package version, and selects the tokenizer associated with GPT-2. A model-specific encoding fixes the vocabulary, merge rules, and special-token IDs used throughout the remaining examples.

Recording the version in a preserved output is useful for reproducibility because tokenizer packages can evolve even when the conceptual workflow remains the same.


In [1]:
from importlib.metadata import version
import tiktoken

In [2]:
print("tiktoken version: ", version("tiktoken"))

tiktoken version:  0.13.0


In [3]:
tokenizer = tiktoken.encoding_for_model("gpt2")

## 2. Encode, decode, and inspect subword pieces

The next examples test three important tokenizer behaviors. First, a sentence containing `<|endoftext|>` is encoded with that marker explicitly allowed and then decoded back to text. Second, an artificial string is encoded to show that BPE does not require every full word to be present in the vocabulary. Finally, each token ID is decoded separately so the subword boundaries become visible.

The round trip demonstrates the practical contract of a tokenizer:

```text
text → token IDs → reconstructed text
```

The individual-token view is intentionally less natural to read; its purpose is to reveal how a single word-like string can be assembled from several learned pieces.


In [4]:
text =(
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    " of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [5]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [6]:
text1 = "Akwirw ier"
ids = tokenizer.encode(text1)
print(ids)

[33901, 86, 343, 86, 220, 959]


In [7]:
# Exercise 1
for id_obj in ids:
    print(tokenizer.decode([id_obj]))

Ak
w
ir
w
 
ier


## 3. Tokenize the chapter corpus

`text-verdict.txt` provides the continuous text used for the language-model examples. The file is opened with UTF-8 encoding, converted into GPT-2 token IDs, and trimmed from token position 50 to create a smaller sample for inspection.

The preserved output reports **5,145 token IDs** for the full text in this run. That number is evidence from the saved notebook state; rerunning with a different file or tokenizer configuration could change it.


In [8]:
with open("text-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [10]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [13]:
enc_sample = enc_text[50:]

## 4. Form next-token prediction pairs

A causal language model learns to predict the token that follows each visible context. For a context window of length \(L\), the input and target sequences are aligned as:

$$
\mathbf{x} = (t_i, t_{i+1}, \ldots, t_{i+L-1}),
\qquad
\mathbf{y} = (t_{i+1}, t_{i+2}, \ldots, t_{i+L})
$$

Each target is therefore the input sequence shifted one token to the left. The notebook prints this relationship first with integer IDs and then with decoded text. Growing the context one token at a time makes the training objective intuitive: use everything seen so far to predict the next token.


In [14]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x:  {x}")
print(f"y:       {y}")

x:  [290, 4920, 2241, 287]
y:       [4920, 2241, 287, 257]


In [15]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"{context} ---> {desired}")

[290] ---> 4920
[290, 4920] ---> 2241
[290, 4920, 2241] ---> 287
[290, 4920, 2241, 287] ---> 257


In [19]:
for i in range(1, context_size+1):
    context = tokenizer.decode(enc_sample[:i])
    desired = tokenizer.decode([enc_sample[i]])
    print(f"{context} ---> {desired}")

 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


## 5. Package sliding windows as a PyTorch dataset

`GPTDatasetV1` converts the complete token stream into reusable training examples. For every starting position, it stores:

- an input chunk of `max_length` tokens;
- a target chunk beginning one token later;
- a new starting position determined by `stride`.

When `stride < max_length`, neighboring examples overlap and expose the model to closely related contexts. When the values are equal, windows are adjacent and non-overlapping. The class implements PyTorch's dataset protocol through `__len__` and `__getitem__`, while `create_dataloader()` adds batching, optional shuffling, worker configuration, and incomplete-batch handling.


In [41]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    """Create sliding-window token pairs for next-token prediction.

    Args:
        txt (str): Source text used to create training sequences.
        tokenizer: Tokenizer that converts text into integer token IDs.
        max_length (int): Number of tokens in each input sequence.
        stride (int): Number of token positions between consecutive windows.

    Attributes:
        input_ids (list[torch.Tensor]): Input token sequences.
        target_ids (list[torch.Tensor]): Sequences shifted one token ahead.
    """

    def __init__(self, txt, tokenizer, max_length, stride):
        """Initialize input-target windows from the source text."""

        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        # The stride controls how much neighboring training windows overlap.
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            # Shift by one token so each position predicts its successor.
            target_chunk = token_ids[i + 1: i+ max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        """Return the number of available input-target windows.

        Returns:
            int: Number of training examples in the dataset.
        """
        return len(self.input_ids)

    def __getitem__(self, idx):
        """Return one aligned input-target training example.

        Args:
            idx (int): Zero-based dataset index.

        Returns:
            tuple[torch.Tensor, torch.Tensor]: Input IDs and next-token targets.
        """
        return self.input_ids[idx], self.target_ids[idx]

In [42]:
def create_dataloader(txt, batch_size=4, max_length=256, 
                      stride=128, shuffle=True, drop_last=True,
                      num_workers=0):
    """Build a PyTorch data loader for GPT-style next-token training.

    Args:
        txt (str): Source text to tokenize.
        batch_size (int): Number of token windows per batch.
        max_length (int): Token count in each input window.
        stride (int): Offset between consecutive windows.
        shuffle (bool): Whether to randomize example order each epoch.
        drop_last (bool): Whether to discard an incomplete final batch.
        num_workers (int): Number of worker processes used for loading.

    Returns:
        DataLoader: Batches of aligned input and target token tensors.
    """
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

## 6. Inspect stride, overlap, and batch shapes

The final experiments call the same data-loading function with several configurations. A stride of one produces strongly overlapping windows; a stride equal to the context length produces adjacent examples; and a larger batch groups several windows into two-dimensional tensors.

Read the input and target tensors row by row. Every target row should equal its input row shifted forward by one token. This invariant matters more than the specific token values because it is what makes next-token prediction possible.

> The examples deliberately keep `shuffle=False` while inspecting alignment. Shuffling is useful during training, but deterministic ordering is clearer when verifying the pipeline.


In [43]:
with open("text-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [44]:
# Exercise 2
dataloader = create_dataloader(raw_text, batch_size=1, max_length=2, stride=2, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]
[tensor([[2885, 1464]]), tensor([[1464, 1807]])]


In [45]:
# Exercise 2
dataloader = create_dataloader(raw_text, batch_size=1, max_length=8, stride=2, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]
[tensor([[ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]]), tensor([[ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]])]


In [46]:
dataloader = create_dataloader(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)
data_iter = iter(dataloader)
inputs, outputs = next(data_iter)
print("Inputs: \n", inputs)
print("Outputs: \n", outputs)

Inputs: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Outputs: 
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Conclusion

This notebook connects text tokenization to the tensor pairs required by a GPT training loop. The preserved examples confirm that the GPT-2 tokenizer can round-trip ordinary text and special tokens, represent unfamiliar strings as smaller pieces, and transform the chapter corpus into aligned context-target windows.

| Component | Role in the pipeline | Preserved evidence |
|---|---|---|
| `tiktoken` GPT-2 encoding | Maps text to token IDs and back | Encoded IDs and reconstructed sentences |
| Context-target shift | Defines next-token supervision | Integer and decoded context arrows |
| `GPTDatasetV1` | Stores sliding input-target windows | Consecutive tensor pairs |
| `DataLoader` | Batches examples for training | Input and output tensors with matching shapes |

The next step is to map these discrete token IDs into continuous vectors. The chapter's embedding notebook performs that transformation with token and positional embedding layers.


## Resources & References

- [Chapter 2 companion code — *Build a Large Language Model (From Scratch)*](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/01_main-chapter-code)
- [OpenAI `tiktoken` repository and educational BPE notes](https://github.com/openai/tiktoken)
- [PyTorch data loading utilities](https://docs.pytorch.org/docs/stable/data.html)
- [Book page — *Build a Large Language Model (From Scratch)*](https://www.manning.com/books/build-a-large-language-model-from-scratch)
